# Test Notebook - Long Straddle Periodico

Validacion de todos los modulos antes del backtest completo.

### Importaciones y configuracion del entorno

Carga de librerias necesarias y configuracion del path para acceder a los modulos personalizados.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.stats import norm
import sys
import os
from dotenv import load_dotenv
from fredapi import Fred
import QuantLib as ql

# Agregar el directorio scripts al path
scripts_path = os.path.join(os.path.dirname(os.getcwd()), 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 200)

## 1. Test Data Loader

In [2]:
from data_loader import get_spy_history, get_vix_history, load_treasury_curve

# Periodo de prueba (5 anios)
START_DATE = '2024-11-01'
END_DATE = '2025-11-01'

In [3]:
# SPY
spy_data = get_spy_history(START_DATE, END_DATE)
print("SPY Data:")
print(f"  Shape: {spy_data.shape}")
print(f"  Rango: {spy_data.index[0]} a {spy_data.index[-1]}")
print(f"  Columnas: {list(spy_data.columns)}")
print()
spy_data.head()

SPY Data:
  Shape: (250, 2)
  Rango: 2024-11-01 a 2025-10-31
  Columnas: ['close_spy', 'q_yield']



,close_spy,q_yield
Date,,
2024-11-01,571.039978,0.012269
2024-11-04,569.809998,0.012295
2024-11-05,576.700012,0.012148
2024-11-06,591.039978,0.011854
2024-11-07,595.609985,0.011763


In [4]:
# Verificar dividend yield (debe estar entre 1-2%)
print(f"Dividend yield - Min: {spy_data['q_yield'].min():.4f}, Max: {spy_data['q_yield'].max():.4f}, Mean: {spy_data['q_yield'].mean():.4f}")

Dividend yield - Min: 0.0084, Max: 0.0150, Mean: 0.0117


In [5]:
# VIX
vix_data = get_vix_history(START_DATE, END_DATE)
print("VIX Data:")
print(f"  Shape: {vix_data.shape}")
print(f"  Rango: {vix_data.index[0]} a {vix_data.index[-1]}")
print()
# Verificar que viene normalizado (valores entre 0.1 y 0.5 aprox)
print(f"VIX (decimal) - Min: {vix_data.min():.4f}, Max: {vix_data.max():.4f}, Mean: {vix_data.mean():.4f}")
vix_data.head()

VIX Data:
  Shape: (250,)
  Rango: 2024-11-01 a 2025-10-31

VIX (decimal) - Min: 0.1277, Max: 0.5233, Mean: 0.1870


Date
2024-11-01    0.2188
2024-11-04    0.2198
2024-11-05    0.2049
2024-11-06    0.1627
2024-11-07    0.1520
Name: close_vix, dtype: float64

In [6]:
# Treasury Curve
treasury_data = load_treasury_curve(START_DATE, END_DATE)
print("Treasury Data:")
print(f"  Shape: {treasury_data.shape}")
print(f"  Rango: {treasury_data.index[0]} a {treasury_data.index[-1]}")
print(f"  Columnas: {list(treasury_data.columns)}")
print()
treasury_data.head()

Treasury Data:
  Shape: (261, 4)
  Rango: 2024-11-01 a 2025-10-31
  Columnas: [30, 90, 180, 365]



,30,90,180,365
2024-11-01,0.0475,0.0461,0.0442,0.0428
2024-11-04,0.0475,0.0465,0.0439,0.0425
2024-11-05,0.0472,0.0464,0.0439,0.0427
2024-11-06,0.0468,0.0464,0.0441,0.0431
2024-11-07,0.0469,0.0463,0.0440,0.0428


In [7]:
# Verificar tasas (deben estar en decimal, ej: 0.05 para 5%)
print("Tasas (ultimo dia):")
print(treasury_data.iloc[-1])

Tasas (ultimo dia):
30     0.0406
90     0.0389
180    0.0379
365    0.0370
Name: 2025-10-31, dtype: float64


## 2. Test Rates Interpolation

In [8]:
from rates import get_risk_free_rate

# Usar la curva de un dia especifico
sample_date = treasury_data.index[100]
curve_row = treasury_data.loc[sample_date]

print(f"Curva del {sample_date}:")
print(curve_row)
print()

# Probar interpolacion para diferentes tenores
test_tenors = [7, 15, 30, 45, 60, 90, 120, 180, 270, 365]

print("Interpolacion:")
for T in test_tenors:
    r = get_risk_free_rate(T, curve_row)
    print(f"  T={T:3d} dias -> r = {r:.4f} ({r*100:.2f}%)")

Curva del 2025-03-21:
30     0.0436
90     0.0433
180    0.0426
365    0.0404
Name: 2025-03-21, dtype: float64

Interpolacion:
  T=  7 dias -> r = 0.0436 (4.36%)
  T= 15 dias -> r = 0.0436 (4.36%)
  T= 30 dias -> r = 0.0436 (4.36%)
  T= 45 dias -> r = 0.0435 (4.35%)
  T= 60 dias -> r = 0.0435 (4.35%)
  T= 90 dias -> r = 0.0433 (4.33%)
  T=120 dias -> r = 0.0431 (4.31%)
  T=180 dias -> r = 0.0426 (4.26%)
  T=270 dias -> r = 0.0415 (4.15%)
  T=365 dias -> r = 0.0404 (4.04%)


## 3. Test Black-Scholes

In [9]:
from black_scholes import black_scholes_merton, calculate_all_greeks

# Parametros de prueba
S = 450.0      # Spot SPY
K = 450.0      # Strike ATM
T = 30/365     # 30 dias
r = 0.05       # 5% tasa
q = 0.015      # 1.5% dividend yield
sigma = 0.18   # 18% volatilidad

print("Parametros:")
print(f"  S = ${S:.2f}")
print(f"  K = ${K:.2f}")
print(f"  T = {T*365:.0f} dias")
print(f"  r = {r*100:.2f}%")
print(f"  q = {q*100:.2f}%")
print(f"  sigma = {sigma*100:.1f}%")
print()

Parametros:
  S = $450.00
  K = $450.00
  T = 30 dias
  r = 5.00%
  q = 1.50%
  sigma = 18.0%



In [10]:
# Pricing
call_price = black_scholes_merton(S, K, T, r, q, sigma, 'call')
put_price = black_scholes_merton(S, K, T, r, q, sigma, 'put')

print("Precios:")
print(f"  Call: ${call_price:.2f}")
print(f"  Put:  ${put_price:.2f}")
print(f"  Straddle: ${call_price + put_price:.2f}")

Precios:
  Call: $9.90
  Put:  $8.61
  Straddle: $18.51


In [11]:
# Greeks
call_greeks = calculate_all_greeks(S, K, T, r, q, sigma, 'call')
put_greeks = calculate_all_greeks(S, K, T, r, q, sigma, 'put')

print("Greeks Call:")
for greek, value in call_greeks.items():
    print(f"  {greek:6s}: {value:+.4f}")

print("\nGreeks Put:")
for greek, value in put_greeks.items():
    print(f"  {greek:6s}: {value:+.4f}")

Greeks Call:
  delta : +0.5318
  gamma : +0.0171
  vega  : +0.5123
  theta : -0.1753
  rho   : +0.1886

Greeks Put:
  delta : -0.4669
  gamma : +0.0171
  vega  : +0.5123
  theta : -0.1324
  rho   : -0.1798


In [12]:
# Verificar paridad put-call: C - P = S*e^(-qT) - K*e^(-rT)
lhs = call_price - put_price
rhs = S * np.exp(-q * T) - K * np.exp(-r * T)

print("Verificacion paridad Put-Call:")
print(f"  C - P = {lhs:.4f}")
print(f"  S*e^(-qT) - K*e^(-rT) = {rhs:.4f}")
print(f"  Diferencia: {abs(lhs - rhs):.6f} (debe ser ~0)")

Verificacion paridad Put-Call:
  C - P = 1.2911
  S*e^(-qT) - K*e^(-rT) = 1.2911
  Diferencia: 0.000000 (debe ser ~0)


## 4. Test Straddle

In [13]:
from straddle import price_straddle, calculate_straddle_greeks

# Pricing
straddle_prices = price_straddle(S, K, T, r, q, sigma)
print("Straddle Pricing:")
print(f"  Call:     ${straddle_prices['call']:.2f}")
print(f"  Put:      ${straddle_prices['put']:.2f}")
print(f"  Straddle: ${straddle_prices['straddle']:.2f}")

Straddle Pricing:
  Call:     $9.90
  Put:      $8.61
  Straddle: $18.51


In [14]:
# Greeks del straddle
straddle_greeks = calculate_straddle_greeks(S, K, T, r, q, sigma)

print("\nStraddle Greeks:")
for greek, value in straddle_greeks.items():
    print(f"  {greek:6s}: {value:+.4f}")

print("\nVerificaciones:")
print(f"  Delta ~ 0 (neutralidad): {abs(straddle_greeks['delta']) < 0.05}")
print(f"  Gamma > 0 (convexidad): {straddle_greeks['gamma'] > 0}")
print(f"  Vega > 0 (long vol): {straddle_greeks['vega'] > 0}")
print(f"  Theta < 0 (time decay): {straddle_greeks['theta'] < 0}")


Straddle Greeks:
  delta : +0.0649
  gamma : +0.0342
  vega  : +1.0247
  theta : -0.3077
  rho   : +0.0088

Verificaciones:
  Delta ~ 0 (neutralidad): False
  Gamma > 0 (convexidad): True
  Vega > 0 (long vol): True
  Theta < 0 (time decay): True


## 5. Test Strategy

In [15]:
from strategy import StraddlePosition, generate_entry_dates, calculate_expiry_date

# Crear posicion de prueba
test_position = StraddlePosition(
    entry_date='2023-01-03',
    expiry_date='2023-02-02',
    strike=400.0,
    entry_price=15.50,
    tenor_days=30,
    current_price=15.50
)

print("Posicion creada:")
print(f"  Entry: {test_position.entry_date}")
print(f"  Expiry: {test_position.expiry_date}")
print(f"  Strike: ${test_position.strike}")
print(f"  Entry price: ${test_position.entry_price}")
print(f"  Status: {test_position.status}")

Posicion creada:
  Entry: 2023-01-03
  Expiry: 2023-02-02
  Strike: $400.0
  Entry price: $15.5
  Status: open


In [16]:
# Probar metodos de la posicion
test_date = '2023-01-15'
print(f"\nEn fecha {test_date}:")
print(f"  Dias a vencimiento: {test_position.days_to_expiry(test_date)}")
print(f"  T (anios): {test_position.time_to_expiry(test_date):.4f}")

# Simular actualizacion MTM
test_position.update_mtm(17.25)
print(f"\nTras update MTM a $17.25:")
print(f"  Current price: ${test_position.current_price}")
print(f"  MTM P&L: ${test_position.mtm_pnl:.2f}")

# Valor intrinseco
spot_at_expiry = 415.0
print(f"\nValor intrinseco con S={spot_at_expiry}:")
print(f"  |S - K| = ${test_position.intrinsic_value(spot_at_expiry):.2f}")


En fecha 2023-01-15:
  Dias a vencimiento: 18
  T (anios): 0.0493

Tras update MTM a $17.25:
  Current price: $17.25
  MTM P&L: $1.75

Valor intrinseco con S=415.0:
  |S - K| = $15.00


In [17]:
# Generar fechas de entrada
trading_days = spy_data.index

entry_dates_monthly = generate_entry_dates(START_DATE, END_DATE, trading_days, 'monthly')
entry_dates_weekly = generate_entry_dates(START_DATE, END_DATE, trading_days, 'weekly')

print(f"Fechas de entrada MONTHLY ({len(entry_dates_monthly)} trades):")
print(entry_dates_monthly[:6], "...")

print(f"\nFechas de entrada WEEKLY ({len(entry_dates_weekly)} trades):")
print(entry_dates_weekly[:6], "...")

Fechas de entrada MONTHLY (12 trades):
['2024-11-01', '2024-12-02', '2025-01-02', '2025-02-03', '2025-03-03', '2025-04-01'] ...

Fechas de entrada WEEKLY (53 trades):
['2024-11-01', '2024-11-04', '2024-11-11', '2024-11-18', '2024-11-25', '2024-12-02'] ...


In [18]:
# Calcular fecha de expiry
sample_entry = entry_dates_monthly[0]
expiry = calculate_expiry_date(sample_entry, 30, trading_days)

print(f"Entry: {sample_entry} + 30 dias -> Expiry: {expiry}")

Entry: 2024-11-01 + 30 dias -> Expiry: 2024-12-02


## 6. Test Backtest Completo

In [19]:
from backtest import run_backtest, prepare_market_data, _calculate_summary
# Consolidar datos de mercado
market_data = prepare_market_data(spy_data, vix_data, treasury_data)

print("Market Data consolidado:")
print(f"  Shape: {market_data.shape}")
print(f"  Columnas: {list(market_data.columns)}")
print(f"  Rango: {market_data.index[0]} a {market_data.index[-1]}")
print()
market_data.head()

Market Data consolidado:
  Shape: (250, 7)
  Columnas: ['close_spy', 'q_yield', 'close_vix', 30, 90, 180, 365]
  Rango: 2024-11-01 a 2025-10-31



,close_spy,q_yield,close_vix,30,90,180,365
Date,,,,,,,
2024-11-01,571.039978,0.012269,0.2188,0.0475,0.0461,0.0442,0.0428
2024-11-04,569.809998,0.012295,0.2198,0.0475,0.0465,0.0439,0.0425
2024-11-05,576.700012,0.012148,0.2049,0.0472,0.0464,0.0439,0.0427
2024-11-06,591.039978,0.011854,0.1627,0.0468,0.0464,0.0441,0.0431
2024-11-07,595.609985,0.011763,0.1520,0.0469,0.0463,0.0440,0.0428


In [20]:
# Verificar que no hay NaN
print("Valores nulos por columna:")
print(market_data.isnull().sum())

Valores nulos por columna:
close_spy    0
q_yield      0
close_vix    0
30           0
90           0
180          0
365          0
dtype: int64


In [21]:
# Generar fechas de entrada con el indice correcto
entry_dates = generate_entry_dates(
    market_data.index[0], 
    market_data.index[-1], 
    market_data.index, 
    'weekly'
)

print(f"Fechas de entrada: {len(entry_dates)} trades")
print(entry_dates[:5], "...")

Fechas de entrada: 53 trades
['2024-11-01', '2024-11-04', '2024-11-11', '2024-11-18', '2024-11-25'] ...


In [22]:
# Ejecutar backtest
print("Ejecutando backtest...")
results = run_backtest(market_data, entry_dates, tenor_days=30)
print("Backtest completado!")

Ejecutando backtest...
Backtest completado!


In [23]:
# Resumen
print("=" * 50)
print("RESUMEN DEL BACKTEST")
print("=" * 50)

for key, value in results.summary.items():
    if isinstance(value, float):
        print(f"  {key:20s}: {value:>10.2f}")
    else:
        print(f"  {key:20s}: {value:>10}")

RESUMEN DEL BACKTEST
  total_pnl_straddle  :    -263.01
  num_trades          :         53
  avg_pnl_per_trade   :      -4.96
  win_rate            :       0.34
  avg_win             :       8.93
  avg_loss            :     -12.11
  best_trade          :      29.20
  worst_trade         :     -24.00
  open_positions      :          0
  total_pnl           :   -2461.69
  max_drawdown        :   -2468.01
  profit_factor       :       0.38


In [24]:
# Grafico de P&L acumulado usando Plotly
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('P&L Acumulado - Long Straddle Mensual (30 días)', 'P&L Diario'),
    vertical_spacing=0.12,
    row_heights=[0.5, 0.5]
)

# P&L acumulado
fig.add_trace(
    go.Scatter(
        x=results.cumulative_pnl_straddle.index,
        y=results.cumulative_pnl_straddle.values,
        mode='lines',
        name='P&L Acumulado',
        line=dict(color='#2E86AB', width=2),
        hovertemplate='<b>Fecha:</b> %{x|%Y-%m-%d}<br><b>P&L Acumulado:</b> $%{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)

# Línea de referencia en 0
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=1)

# P&L diario
fig.add_trace(
    go.Scatter(
        x=results.daily_pnl_straddle.index,
        y=results.daily_pnl_straddle.values,
        mode='lines',
        name='P&L Diario',
        line=dict(color='#06A77D', width=1),
        fill='tozeroy',
        fillcolor='rgba(6, 167, 125, 0.2)',
        hovertemplate='<b>Fecha:</b> %{x|%Y-%m-%d}<br><b>P&L Diario:</b> $%{y:.2f}<extra></extra>'
    ),
    row=2, col=1
)

# Línea de referencia en 0
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=2, col=1)

# Actualizar layout
fig.update_xaxes(title_text="Fecha", row=2, col=1)
fig.update_yaxes(title_text="P&L ($)", row=1, col=1)
fig.update_yaxes(title_text="P&L ($)", row=2, col=1)

fig.update_layout(
    height=700,
    showlegend=True,
    hovermode='x unified',
    template='plotly_white',
    title_text="Análisis de P&L - Estrategia Long Straddle",
    title_x=0.5,
    title_font_size=16
)

fig.show()

In [25]:
# Detalle de trades
print("Primeros 5 trades:")
print("-" * 80)

for i, trade in enumerate(results.trades[:5]):
    print(f"Trade {i+1}:")
    print(f"  Entry: {trade.entry_date} | Expiry: {trade.expiry_date}")
    print(f"  Strike: ${trade.strike} | Entry Price: ${trade.entry_price:.2f}")
    exit_price_str = f"${trade.exit_price:.2f}" if trade.exit_price else "N/A"
    print(f"  Exit Price: {exit_price_str}")
    pnl_value = trade.realized_pnl if trade.realized_pnl else trade.mtm_pnl
    print(f"  P&L: ${pnl_value:.2f}")
    print(f"  Status: {trade.status}")
    print()

Primeros 5 trades:
--------------------------------------------------------------------------------
Trade 1:
  Entry: 2024-11-01 | Expiry: 2024-12-02
  Strike: $571 | Entry Price: $28.54
  Exit Price: $32.63
  P&L: $4.09
  Status: closed

Trade 2:
  Entry: 2024-11-04 | Expiry: 2024-12-04
  Strike: $570 | Entry Price: $28.60
  Exit Price: $37.66
  P&L: $9.06
  Status: closed

Trade 3:
  Entry: 2024-11-11 | Expiry: 2024-12-11
  Strike: $599 | Entry Price: $20.49
  Exit Price: $8.46
  P&L: $-12.03
  Status: closed

Trade 4:
  Entry: 2024-11-18 | Expiry: 2024-12-18
  Strike: $588 | Entry Price: $20.96
  Exit Price: $1.72
  P&L: $-19.24
  Status: closed

Trade 5:
  Entry: 2024-11-25 | Expiry: 2024-12-26
  Strike: $598 | Entry Price: $19.94
  Exit Price: $3.34
  P&L: $-16.60
  Status: closed



In [26]:
# Distribucion de P&L por trade usando Plotly
closed_trades = [t for t in results.trades if t.status == 'closed']
trade_pnls = [t.realized_pnl for t in closed_trades]

fig = go.Figure()

# Histograma
fig.add_trace(go.Histogram(
    x=trade_pnls,
    nbinsx=20,
    name='P&L',
    marker_color='#A23B72',
    opacity=0.75,
    hovertemplate='<b>Rango P&L:</b> %{x:.2f}<br><b>Frecuencia:</b> %{y}<extra></extra>'
))

# Línea vertical en 0
fig.add_vline(x=0, line_dash="dash", line_color="red", line_width=2, annotation_text="Break-even")

# Línea vertical en la media
mean_pnl = np.mean(trade_pnls)
fig.add_vline(
    x=mean_pnl, 
    line_dash="solid", 
    line_color="green", 
    line_width=2,
    annotation_text=f"Media: ${mean_pnl:.2f}",
    annotation_position="top"
)

fig.update_layout(
    title='Distribución de P&L por Trade',
    title_x=0.5,
    xaxis_title='P&L ($)',
    yaxis_title='Frecuencia',
    template='plotly_white',
    height=500,
    showlegend=False,
    hovermode='x'
)

fig.show()

## 7. Comparacion con SPY (Benchmark)

In [27]:
# Comparar con buy & hold SPY usando Plotly
spy_returns = market_data['close_spy'].pct_change().fillna(0)
spy_cumulative = (1 + spy_returns).cumprod() - 1

# Normalizar P&L del straddle como % del capital inicial
# Asumiendo que invertimos el precio del primer straddle
first_trade_cost = results.trades[0].entry_price
straddle_returns = results.cumulative_pnl_straddle / first_trade_cost

fig = go.Figure()

# SPY Buy & Hold
fig.add_trace(go.Scatter(
    x=spy_cumulative.index,
    y=spy_cumulative.values * 100,
    mode='lines',
    name='SPY Buy & Hold',
    line=dict(color='#F18F01', width=2),
    hovertemplate='<b>Fecha:</b> %{x|%Y-%m-%d}<br><b>Retorno SPY:</b> %{y:.2f}%<extra></extra>'
))

# Long Straddle
fig.add_trace(go.Scatter(
    x=straddle_returns.index,
    y=straddle_returns.values * 100,
    mode='lines',
    name='Long Straddle',
    line=dict(color='#6A4C93', width=2),
    hovertemplate='<b>Fecha:</b> %{x|%Y-%m-%d}<br><b>Retorno Straddle:</b> %{y:.2f}%<extra></extra>'
))

# Línea de referencia en 0
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

# Añadir estadísticas al final
spy_final_return = spy_cumulative.iloc[-1] * 100
straddle_final_return = straddle_returns.iloc[-1] * 100

fig.add_annotation(
    x=0.02, y=0.98,
    xref="paper", yref="paper",
    text=f"<b>Retorno Final:</b><br>SPY: {spy_final_return:.2f}%<br>Straddle: {straddle_final_return:.2f}%",
    showarrow=False,
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
    align="left",
    font=dict(size=11)
)

fig.update_layout(
    title='Comparación: Long Straddle vs SPY Buy & Hold',
    title_x=0.5,
    xaxis_title='Fecha',
    yaxis_title='Retorno (%)',
    template='plotly_white',
    height=600,
    hovermode='x unified',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1
    )
)

fig.show()

## 8. Verificaciones Finales

In [28]:
print("VERIFICACIONES:")
print("=" * 50)

# 1. Numero de trades
expected_trades = len(entry_dates)
actual_trades = len(results.trades)
print(f"1. Trades esperados: {expected_trades}, Actual: {actual_trades} -> {'OK' if expected_trades == actual_trades else 'ERROR'}")

# 2. Todos los trades cerrados excepto los ultimos
open_trades = [t for t in results.trades if t.status == 'open']
print(f"2. Trades abiertos al final: {len(open_trades)} -> {'OK (puede haber 1)' if len(open_trades) <= 1 else 'REVISAR'}")

# 3. P&L consistente
total_from_trades = sum(t.realized_pnl for t in closed_trades)
print(f"3. P&L total desde trades: ${total_from_trades:.2f}")
print(f"   P&L total desde summary: ${results.summary['total_pnl']:.2f}")

# 4. Strikes son ATM (enteros)
strikes = [t.strike for t in results.trades]
all_integer = all(s == int(s) for s in strikes)
print(f"4. Todos los strikes son enteros (ATM): {'OK' if all_integer else 'ERROR'}")

# 5. Entry prices razonables (entre $5 y $50 para SPY)
entry_prices = [t.entry_price for t in results.trades]
prices_ok = all(5 < p < 50 for p in entry_prices)
print(f"5. Entry prices en rango razonable ($5-$50): {'OK' if prices_ok else 'REVISAR'}")
print(f"   Min: ${min(entry_prices):.2f}, Max: ${max(entry_prices):.2f}")

VERIFICACIONES:
1. Trades esperados: 53, Actual: 53 -> OK
2. Trades abiertos al final: 0 -> OK (puede haber 1)
3. P&L total desde trades: $-263.01
   P&L total desde summary: $-2461.69
4. Todos los strikes son enteros (ATM): OK
5. Entry prices en rango razonable ($5-$50): REVISAR
   Min: $18.41, Max: $54.03


In [29]:
# Verificar que la funcion _calculate_summary funciona correctamente
# Esta funcion ahora espera parametros separados para straddle, hedge y totales
print("Verificando metricas del summary...")
print(f"Total P&L Straddle: ${results.summary['total_pnl_straddle']:.2f}")
print(f"Total P&L: ${results.summary['total_pnl']:.2f}")
print(f"Max Drawdown: ${results.summary['max_drawdown']:.2f}")
print(f"Profit Factor: {results.summary['profit_factor']:.2f}")

Verificando metricas del summary...
Total P&L Straddle: $-263.01
Total P&L: $-2461.69
Max Drawdown: $-2468.01
Profit Factor: 0.38


## 9. Evolucion de las Griegas por Straddle

Visualizacion de la evolucion temporal de las griegas (Delta, Gamma, Vega, Theta, Rho) para un straddle especifico seleccionado.

In [36]:
def calculate_greeks_evolution(straddle_idx, results, market_data):
    """
    Calcula la evolucion de las griegas para un straddle especifico.
    
    Parameters
    ----------
    straddle_idx : int
        Indice del straddle (0-based). Ej: 0 para el primer straddle, 34 para el straddle #35
    results : BacktestResult
        Resultados del backtest
    market_data : pd.DataFrame
        Datos de mercado consolidados
        
    Returns
    -------
    pd.DataFrame
        DataFrame con columnas: fecha, delta, gamma, vega, theta, rho
    """
    from straddle import calculate_straddle_greeks
    from rates import get_risk_free_rate
    
    # Validar indice
    if straddle_idx < 0 or straddle_idx >= len(results.trades):
        raise ValueError(f"Indice {straddle_idx} fuera de rango. Hay {len(results.trades)} straddles.")
    
    # Obtener el trade seleccionado
    trade = results.trades[straddle_idx]
    
    # Rango de fechas de vida del straddle - convertir a string para comparar
    entry_date = trade.entry_date
    expiry_date = trade.expiry_date
    
    # Asegurar que el índice esté en formato datetime
    market_data_copy = market_data.copy()
    if not isinstance(market_data_copy.index, pd.DatetimeIndex):
        market_data_copy.index = pd.to_datetime(market_data_copy.index)
    
    # Convertir las fechas del trade a Timestamp para comparar
    entry_ts = pd.Timestamp(entry_date)
    expiry_ts = pd.Timestamp(expiry_date)
    
    # Filtrar dias de trading en el rango
    trading_days_in_range = market_data_copy.loc[
        (market_data_copy.index >= entry_ts) & (market_data_copy.index <= expiry_ts)
    ].index
    
    # Calcular griegas para cada dia
    greeks_data = []
    
    for date in trading_days_in_range:
        row = market_data_copy.loc[date]
        S = row['close_spy']
        sigma = row['close_vix']
        q = row['q_yield']
        
        # Tiempo hasta vencimiento
        T = trade.time_to_expiry(str(date.date()))
        
        if T > 0:
            # Obtener tasa risk-free
            days_remaining = trade.days_to_expiry(str(date.date()))
            curve_row = {
                30: row[30],
                90: row[90],
                180: row[180],
                365: row[365]
            }
            r = get_risk_free_rate(days_remaining, curve_row)
            
            # Calcular griegas
            greeks = calculate_straddle_greeks(S, trade.strike, T, r, q, sigma)
            
            greeks_data.append({
                'date': date,
                'delta': greeks['delta'],
                'gamma': greeks['gamma'],
                'vega': greeks['vega'],
                'theta': greeks['theta'],
                'rho': greeks['rho'],
                'days_to_expiry': days_remaining,
                'spot': S,
                'strike': trade.strike
            })
        else:
            # Al vencimiento, las griegas se colapsan
            greeks_data.append({
                'date': date,
                'delta': 1.0 if S > trade.strike else -1.0 if S < trade.strike else 0.0,
                'gamma': 0.0,
                'vega': 0.0,
                'theta': 0.0,
                'rho': 0.0,
                'days_to_expiry': 0,
                'spot': S,
                'strike': trade.strike
            })
    
    return pd.DataFrame(greeks_data).set_index('date')


def plot_greeks_evolution(straddle_idx, results, market_data):
    """
    Grafica la evolucion de las griegas para un straddle especifico usando Plotly.
    
    Parameters
    ----------
    straddle_idx : int
        Indice del straddle (0-based)
    results : BacktestResult
        Resultados del backtest
    market_data : pd.DataFrame
        Datos de mercado
    """
    # Calcular griegas
    greeks_df = calculate_greeks_evolution(straddle_idx, results, market_data)
    trade = results.trades[straddle_idx]
    
    # Crear subplots para cada griega
    fig = make_subplots(
        rows=5, cols=1,
        subplot_titles=('Delta', 'Gamma', 'Vega', 'Theta', 'Rho'),
        vertical_spacing=0.05,
        row_heights=[0.2, 0.2, 0.2, 0.2, 0.2]
    )
    
    # Colores para cada griega
    colors = {
        'delta': '#2E86AB',
        'gamma': '#A23B72',
        'vega': '#F18F01',
        'theta': '#C73E1D',
        'rho': '#6A4C93'
    }
    
    # Delta
    fig.add_trace(
        go.Scatter(
            x=greeks_df.index,
            y=greeks_df['delta'],
            mode='lines',
            name='Delta',
            line=dict(color=colors['delta'], width=2),
            hovertemplate='<b>Fecha:</b> %{x|%Y-%m-%d}<br><b>Delta:</b> %{y:.4f}<br><extra></extra>'
        ),
        row=1, col=1
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=1)
    
    # Gamma
    fig.add_trace(
        go.Scatter(
            x=greeks_df.index,
            y=greeks_df['gamma'],
            mode='lines',
            name='Gamma',
            line=dict(color=colors['gamma'], width=2),
            hovertemplate='<b>Fecha:</b> %{x|%Y-%m-%d}<br><b>Gamma:</b> %{y:.4f}<br><extra></extra>'
        ),
        row=2, col=1
    )
    
    # Vega
    fig.add_trace(
        go.Scatter(
            x=greeks_df.index,
            y=greeks_df['vega'],
            mode='lines',
            name='Vega',
            line=dict(color=colors['vega'], width=2),
            hovertemplate='<b>Fecha:</b> %{x|%Y-%m-%d}<br><b>Vega:</b> %{y:.4f}<br><extra></extra>'
        ),
        row=3, col=1
    )
    
    # Theta
    fig.add_trace(
        go.Scatter(
            x=greeks_df.index,
            y=greeks_df['theta'],
            mode='lines',
            name='Theta',
            line=dict(color=colors['theta'], width=2),
            hovertemplate='<b>Fecha:</b> %{x|%Y-%m-%d}<br><b>Theta:</b> %{y:.4f}<br><extra></extra>'
        ),
        row=4, col=1
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=4, col=1)
    
    # Rho
    fig.add_trace(
        go.Scatter(
            x=greeks_df.index,
            y=greeks_df['rho'],
            mode='lines',
            name='Rho',
            line=dict(color=colors['rho'], width=2),
            hovertemplate='<b>Fecha:</b> %{x|%Y-%m-%d}<br><b>Rho:</b> %{y:.4f}<br><extra></extra>'
        ),
        row=5, col=1
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=5, col=1)
    
    # Actualizar ejes
    fig.update_xaxes(title_text="Fecha", row=5, col=1)
    for i in range(1, 6):
        fig.update_yaxes(title_text="Valor", row=i, col=1)
    
    # Layout
    fig.update_layout(
        height=1200,
        showlegend=False,
        hovermode='x unified',
        template='plotly_white',
        title_text=f"Evolucion de Griegas - Straddle #{straddle_idx + 1}<br>" + 
                   f"<sub>Entry: {trade.entry_date} | Expiry: {trade.expiry_date} | Strike: ${trade.strike:.0f} | " +
                   f"Entry Price: ${trade.entry_price:.2f}</sub>",
        title_x=0.5,
        title_font_size=16
    )
    
    fig.show()
    
    # Mostrar informacion del straddle
    pnl_value = trade.realized_pnl if trade.realized_pnl is not None else trade.mtm_pnl
    print(f"\n{'='*80}")
    print(f"INFORMACION DEL STRADDLE #{straddle_idx + 1}")
    print(f"{'='*80}")
    print(f"Entry Date:    {trade.entry_date}")
    print(f"Expiry Date:   {trade.expiry_date}")
    print(f"Strike:        ${trade.strike:.2f}")
    print(f"Entry Price:   ${trade.entry_price:.2f}")
    print(f"Exit Price:    ${trade.exit_price:.2f}" if trade.exit_price else "Exit Price:    N/A (Open)")
    print(f"P&L:           ${pnl_value:.2f}")
    print(f"Status:        {trade.status}")
    print(f"{'='*80}\n")
    
    return greeks_df

In [37]:
# Ejemplo: Visualizar la evolucion de las griegas del primer straddle (indice 0)
# Cambia el indice para ver otros straddles (0 a 52 para este backtest de 53 trades)

STRADDLE_INDEX = 0  # Cambiar este valor para seleccionar otro straddle

print(f"Visualizando evolucion de griegas para el Straddle #{STRADDLE_INDEX + 1}")
print(f"Total de straddles disponibles: {len(results.trades)}")
print()

greeks_df = plot_greeks_evolution(STRADDLE_INDEX, results, market_data)

Visualizando evolucion de griegas para el Straddle #1
Total de straddles disponibles: 53




INFORMACION DEL STRADDLE #1
Entry Date:    2024-11-01
Expiry Date:   2024-12-02
Strike:        $571.00
Entry Price:   $28.54
Exit Price:    $32.63
P&L:           $4.09
Status:        closed



In [38]:
# Ejemplos adicionales: Encontrar y visualizar straddles interesantes

# Encontrar el mejor y peor trade
closed_trades_with_idx = [(i, t) for i, t in enumerate(results.trades) if t.status == 'closed']
best_idx, best_trade = max(closed_trades_with_idx, key=lambda x: x[1].realized_pnl)
worst_idx, worst_trade = min(closed_trades_with_idx, key=lambda x: x[1].realized_pnl)

print("Straddles mas interesantes para analizar:")
print("=" * 80)
print(f"\nMEJOR TRADE (Straddle #{best_idx + 1}):")
print(f"  Entry: {best_trade.entry_date} -> Expiry: {best_trade.expiry_date}")
print(f"  P&L: ${best_trade.realized_pnl:.2f}")
print(f"  Strike: ${best_trade.strike:.0f} | Entry Price: ${best_trade.entry_price:.2f}")

print(f"\nPEOR TRADE (Straddle #{worst_idx + 1}):")
print(f"  Entry: {worst_trade.entry_date} -> Expiry: {worst_trade.expiry_date}")
print(f"  P&L: ${worst_trade.realized_pnl:.2f}")
print(f"  Strike: ${worst_trade.strike:.0f} | Entry Price: ${worst_trade.entry_price:.2f}")

print("\n" + "=" * 80)
print("\nPara visualizar alguno de estos straddles, cambia STRADDLE_INDEX arriba a:")
print(f"  - {best_idx} para ver el mejor trade")
print(f"  - {worst_idx} para ver el peor trade")

Straddles mas interesantes para analizar:

MEJOR TRADE (Straddle #26):
  Entry: 2025-04-21 -> Expiry: 2025-05-21
  P&L: $29.20
  Strike: $514 | Entry Price: $39.66

PEOR TRADE (Straddle #23):
  Entry: 2025-03-31 -> Expiry: 2025-04-30
  P&L: $-24.00
  Strike: $559 | Entry Price: $28.46


Para visualizar alguno de estos straddles, cambia STRADDLE_INDEX arriba a:
  - 25 para ver el mejor trade
  - 22 para ver el peor trade


In [42]:
# Resumen de Insights: Griegas vs P&L
print("=" * 80)
print("RESUMEN DE INSIGHTS: GRIEGAS DEL PORTFOLIO vs P&L")
print("=" * 80)
print()

# 1. Promedios de griegas del portfolio
print("1. GRIEGAS PROMEDIO DEL PORTFOLIO (No Hedged)")
print("-" * 80)
print(f"   Delta promedio:    {portfolio_greeks['delta'].mean():+.2f}")
print(f"   Gamma promedio:    {portfolio_greeks['gamma'].mean():+.2f}")
print(f"   Vega promedio:     {portfolio_greeks['vega'].mean():+.2f}")
print(f"   Theta promedio:    {portfolio_greeks['theta'].mean():+.2f}")
print(f"   Rho promedio:      {portfolio_greeks['rho'].mean():+.2f}")
print(f"   Posiciones activas (promedio): {portfolio_greeks['num_positions'].mean():.1f}")
print()

# 2. Rangos de griegas
print("2. RANGOS DE GRIEGAS")
print("-" * 80)
print(f"   Delta:  [{portfolio_greeks['delta'].min():+.2f}, {portfolio_greeks['delta'].max():+.2f}]")
print(f"   Gamma:  [{portfolio_greeks['gamma'].min():+.2f}, {portfolio_greeks['gamma'].max():+.2f}]")
print(f"   Vega:   [{portfolio_greeks['vega'].min():+.2f}, {portfolio_greeks['vega'].max():+.2f}]")
print(f"   Theta:  [{portfolio_greeks['theta'].min():+.2f}, {portfolio_greeks['theta'].max():+.2f}]")
print()

# 3. Correlaciones mas significativas
print("3. CORRELACIONES MAS SIGNIFICATIVAS (Cambios Diarios)")
print("-" * 80)
corrs = {}
for greek in ['delta_change', 'gamma_change', 'vega_change', 'theta_change', 'rho_change']:
    corrs[greek] = abs(combined_df['pnl_change'].corr(combined_df[greek]))

sorted_corrs = sorted(corrs.items(), key=lambda x: x[1], reverse=True)
for greek, corr_value in sorted_corrs:
    greek_name = greek.replace('_change', '').capitalize()
    actual_corr = combined_df['pnl_change'].corr(combined_df[greek])
    print(f"   {greek_name:8s}: {actual_corr:+.4f}  (|r| = {corr_value:.4f})")
print()

# 4. Observaciones clave
print("4. OBSERVACIONES CLAVE")
print("-" * 80)
print(f"   - El portfolio tiene un Delta promedio de {portfolio_greeks['delta'].mean():+.2f},")
print(f"     {'cercano a 0 (market neutral)' if abs(portfolio_greeks['delta'].mean()) < 5 else 'indicando exposicion direccional'}")
print(f"   - Gamma positivo ({portfolio_greeks['gamma'].mean():+.2f}) indica convexidad favorable")
print(f"   - Vega positivo ({portfolio_greeks['vega'].mean():+.2f}) indica beneficio de incrementos en volatilidad")
print(f"   - Theta negativo ({portfolio_greeks['theta'].mean():+.2f}) indica decay temporal (costo de carry)")
print()

# 5. Nota sobre estrategia hedged
print("5. PROXIMOS PASOS")
print("-" * 80)
print("   NOTA: Este analisis muestra la estrategia NO HEDGED (Long Straddle puro).")
print("   ")
print("   Para implementar la estrategia HEDGED:")
print("   - Se agregaria delta hedging mediante posiciones en SPY")
print("   - El objetivo seria mantener Delta del portfolio cerca de 0")
print("   - Se calcularian griegas adicionales del hedge (posicion en SPY)")
print("   - Se compararian resultados de P&L entre estrategia hedged vs no hedged")
print()
print("=" * 80)

RESUMEN DE INSIGHTS: GRIEGAS DEL PORTFOLIO vs P&L

1. GRIEGAS PROMEDIO DEL PORTFOLIO (No Hedged)
--------------------------------------------------------------------------------
   Delta promedio:    +0.94
   Gamma promedio:    +0.12
   Vega promedio:     +3.31
   Theta promedio:    -1.98
   Rho promedio:      +0.14
   Posiciones activas (promedio): 4.5

2. RANGOS DE GRIEGAS
--------------------------------------------------------------------------------
   Delta:  [-3.32, +3.93]
   Gamma:  [+0.00, +0.28]
   Vega:   [+0.00, +4.63]
   Theta:  [-5.92, +0.00]

3. CORRELACIONES MAS SIGNIFICATIVAS (Cambios Diarios)
--------------------------------------------------------------------------------


NameError: name 'combined_df' is not defined

In [43]:
# Visualizacion 2: Scatter plots de Griegas vs P&L Diario
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Delta Change vs P&L Change',
        'Gamma Change vs P&L Change',
        'Vega Change vs P&L Change',
        'Theta Change vs P&L Change'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# Delta vs P&L
fig.add_trace(
    go.Scatter(
        x=combined_df['delta_change'],
        y=combined_df['pnl_change'],
        mode='markers',
        marker=dict(
            size=6,
            color=combined_df['pnl_change'],
            colorscale='RdYlGn',
            showscale=False,
            line=dict(width=0.5, color='DarkSlateGrey')
        ),
        name='Delta',
        hovertemplate='<b>Delta Δ:</b> %{x:.2f}<br><b>P&L Δ:</b> $%{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)

# Gamma vs P&L
fig.add_trace(
    go.Scatter(
        x=combined_df['gamma_change'],
        y=combined_df['pnl_change'],
        mode='markers',
        marker=dict(
            size=6,
            color=combined_df['pnl_change'],
            colorscale='RdYlGn',
            showscale=False,
            line=dict(width=0.5, color='DarkSlateGrey')
        ),
        name='Gamma',
        hovertemplate='<b>Gamma Δ:</b> %{x:.2f}<br><b>P&L Δ:</b> $%{y:.2f}<extra></extra>'
    ),
    row=1, col=2
)

# Vega vs P&L
fig.add_trace(
    go.Scatter(
        x=combined_df['vega_change'],
        y=combined_df['pnl_change'],
        mode='markers',
        marker=dict(
            size=6,
            color=combined_df['pnl_change'],
            colorscale='RdYlGn',
            showscale=False,
            line=dict(width=0.5, color='DarkSlateGrey')
        ),
        name='Vega',
        hovertemplate='<b>Vega Δ:</b> %{x:.2f}<br><b>P&L Δ:</b> $%{y:.2f}<extra></extra>'
    ),
    row=2, col=1
)

# Theta vs P&L
fig.add_trace(
    go.Scatter(
        x=combined_df['theta_change'],
        y=combined_df['pnl_change'],
        mode='markers',
        marker=dict(
            size=6,
            color=combined_df['pnl_change'],
            colorscale='RdYlGn',
            showscale=True,
            colorbar=dict(title="P&L Δ ($)", x=1.12),
            line=dict(width=0.5, color='DarkSlateGrey')
        ),
        name='Theta',
        hovertemplate='<b>Theta Δ:</b> %{x:.2f}<br><b>P&L Δ:</b> $%{y:.2f}<extra></extra>'
    ),
    row=2, col=2
)

# Añadir lineas de referencia en 0
for row in [1, 2]:
    for col in [1, 2]:
        fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=row, col=col)
        fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5, row=row, col=col)

# Actualizar ejes
fig.update_xaxes(title_text="Delta Change", row=1, col=1)
fig.update_xaxes(title_text="Gamma Change", row=1, col=2)
fig.update_xaxes(title_text="Vega Change", row=2, col=1)
fig.update_xaxes(title_text="Theta Change", row=2, col=2)

for row in [1, 2]:
    for col in [1, 2]:
        fig.update_yaxes(title_text="P&L Change ($)", row=row, col=col)

# Layout
fig.update_layout(
    height=800,
    showlegend=False,
    template='plotly_white',
    title_text='Relacion entre Cambios en Griegas y Cambios en P&L - No Hedged',
    title_x=0.5,
    title_font_size=16
)

fig.show()

# Calcular y mostrar estadisticas de correlacion
print("\nAnalisis de Correlacion Detallado:")
print("=" * 70)
for greek in ['delta_change', 'gamma_change', 'vega_change', 'theta_change', 'rho_change']:
    corr = combined_df['pnl_change'].corr(combined_df[greek])
    greek_name = greek.replace('_change', '').capitalize()
    print(f"{greek_name:8s} vs P&L: {corr:+.4f}")

NameError: name 'combined_df' is not defined

In [52]:
# Analisis de correlacion entre cambios en griegas y cambios en P&L
# Calcular cambios diarios (deltas)
pnl_changes = results.daily_pnl_straddle
greeks_changes = portfolio_greeks[['delta', 'gamma', 'vega', 'theta', 'rho']].diff()

# Combinar en un solo DataFrame para analisis
combined_df = pd.DataFrame({
    'pnl_change': pnl_changes,
    'delta_change': greeks_changes['delta'],
    'gamma_change': greeks_changes['gamma'],
    'vega_change': greeks_changes['vega'],
    'theta_change': greeks_changes['theta'],
    'rho_change': greeks_changes['rho']
}).dropna()

# Calcular matriz de correlacion
correlation_matrix = combined_df.corr()

print("Matriz de Correlacion: Cambios en P&L vs Cambios en Griegas")
print("=" * 70)
print(correlation_matrix[['pnl_change']].round(3))
print()

# Visualizacion: Heatmap de correlacion
fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=correlation_matrix.values.round(3),
    texttemplate='%{text}',
    textfont={"size": 12},
    colorbar=dict(title="Correlacion")
))

fig.update_layout(
    title='Matriz de Correlacion: Cambios Diarios en P&L y Griegas',
    title_x=0.5,
    width=700,
    height=600,
    template='plotly_white'
)

fig.show()

# Estadisticas descriptivas
print("\nEstadisticas de cambios diarios:")
print("=" * 70)
print(combined_df.describe().round(3))

Matriz de Correlacion: Cambios en P&L vs Cambios en Griegas
              pnl_change
pnl_change           NaN
delta_change         NaN
gamma_change         NaN
vega_change          NaN
theta_change         NaN
rho_change           NaN




Estadisticas de cambios diarios:
       pnl_change  delta_change  gamma_change  vega_change  theta_change  rho_change
count         0.0           0.0           0.0          0.0           0.0         0.0
mean          NaN           NaN           NaN          NaN           NaN         NaN
std           NaN           NaN           NaN          NaN           NaN         NaN
min           NaN           NaN           NaN          NaN           NaN         NaN
25%           NaN           NaN           NaN          NaN           NaN         NaN
50%           NaN           NaN           NaN          NaN           NaN         NaN
75%           NaN           NaN           NaN          NaN           NaN         NaN
max           NaN           NaN           NaN          NaN           NaN         NaN


In [45]:
# Visualizacion 1: Griegas del Portfolio vs P&L Acumulado
fig = make_subplots(
    rows=5, cols=1,
    subplot_titles=(
        'P&L Acumulado',
        'Delta del Portfolio',
        'Gamma del Portfolio',
        'Vega del Portfolio',
        'Theta del Portfolio'
    ),
    vertical_spacing=0.05,
    row_heights=[0.25, 0.1875, 0.1875, 0.1875, 0.1875],
    specs=[
        [{"secondary_y": True}],
        [{"secondary_y": False}],
        [{"secondary_y": False}],
        [{"secondary_y": False}],
        [{"secondary_y": False}]
    ]
)

# Colores
colors_greeks = {
    'pnl': '#2E86AB',
    'positions': '#888888',
    'delta': '#2E86AB',
    'gamma': '#A23B72',
    'vega': '#F18F01',
    'theta': '#C73E1D'
}

# Row 1: P&L Acumulado y Numero de Posiciones
fig.add_trace(
    go.Scatter(
        x=results.cumulative_pnl_straddle.index,
        y=results.cumulative_pnl_straddle.values,
        mode='lines',
        name='P&L Acumulado',
        line=dict(color=colors_greeks['pnl'], width=2),
        hovertemplate='<b>P&L:</b> $%{y:.2f}<extra></extra>'
    ),
    row=1, col=1, secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=portfolio_greeks['num_positions'],
        mode='lines',
        name='Posiciones Activas',
        line=dict(color=colors_greeks['positions'], width=1, dash='dot'),
        hovertemplate='<b>Posiciones:</b> %{y}<extra></extra>'
    ),
    row=1, col=1, secondary_y=True
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=1)

# Row 2: Delta
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=portfolio_greeks['delta'],
        mode='lines',
        name='Delta',
        line=dict(color=colors_greeks['delta'], width=2),
        hovertemplate='<b>Delta:</b> %{y:.2f}<extra></extra>',
        fill='tozeroy',
        fillcolor=f'rgba(46, 134, 171, 0.2)'
    ),
    row=2, col=1
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=2, col=1)

# Row 3: Gamma
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=portfolio_greeks['gamma'],
        mode='lines',
        name='Gamma',
        line=dict(color=colors_greeks['gamma'], width=2),
        hovertemplate='<b>Gamma:</b> %{y:.2f}<extra></extra>',
        fill='tozeroy',
        fillcolor=f'rgba(162, 59, 114, 0.2)'
    ),
    row=3, col=1
)

# Row 4: Vega
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=portfolio_greeks['vega'],
        mode='lines',
        name='Vega',
        line=dict(color=colors_greeks['vega'], width=2),
        hovertemplate='<b>Vega:</b> %{y:.2f}<extra></extra>',
        fill='tozeroy',
        fillcolor=f'rgba(241, 143, 1, 0.2)'
    ),
    row=4, col=1
)

# Row 5: Theta
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=portfolio_greeks['theta'],
        mode='lines',
        name='Theta',
        line=dict(color=colors_greeks['theta'], width=2),
        hovertemplate='<b>Theta:</b> %{y:.2f}<extra></extra>',
        fill='tozeroy',
        fillcolor=f'rgba(199, 62, 29, 0.2)'
    ),
    row=5, col=1
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=5, col=1)

# Actualizar ejes
fig.update_xaxes(title_text="Fecha", row=5, col=1)
fig.update_yaxes(title_text="P&L ($)", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="# Posiciones", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="Delta", row=2, col=1)
fig.update_yaxes(title_text="Gamma", row=3, col=1)
fig.update_yaxes(title_text="Vega", row=4, col=1)
fig.update_yaxes(title_text="Theta", row=5, col=1)

# Layout
fig.update_layout(
    height=1400,
    showlegend=True,
    hovermode='x unified',
    template='plotly_white',
    title_text='Griegas del Portfolio Agregado vs P&L - Estrategia No Hedged',
    title_x=0.5,
    title_font_size=16,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig.show()

In [46]:
def calculate_portfolio_greeks_timeseries(results, market_data):
    """
    Calcula las griegas agregadas del portfolio para cada dia de trading.
    
    Suma las griegas de todas las posiciones activas en cada fecha.
    
    Parameters
    ----------
    results : BacktestResult
        Resultados del backtest
    market_data : pd.DataFrame
        Datos de mercado consolidados
        
    Returns
    -------
    pd.DataFrame
        DataFrame con columnas: delta, gamma, vega, theta, rho, num_positions
    """
    from straddle import calculate_straddle_greeks
    from rates import get_risk_free_rate
    
    # Asegurar que el índice esté en formato datetime
    market_data_copy = market_data.copy()
    if not isinstance(market_data_copy.index, pd.DatetimeIndex):
        market_data_copy.index = pd.to_datetime(market_data_copy.index)
    
    # Inicializar diccionario para guardar griegas diarias
    daily_greeks = {
        'delta': [],
        'gamma': [],
        'vega': [],
        'theta': [],
        'rho': [],
        'num_positions': [],
        'date': []
    }
    
    # Para cada dia de trading
    for date in market_data_copy.index:
        date_str = str(date.date())
        
        # Encontrar todas las posiciones activas en esta fecha
        active_positions = [
            trade for trade in results.trades
            if trade.entry_date <= date_str <= trade.expiry_date
        ]
        
        # Sumar griegas de todas las posiciones activas
        total_delta = 0.0
        total_gamma = 0.0
        total_vega = 0.0
        total_theta = 0.0
        total_rho = 0.0
        
        if active_positions:
            row = market_data_copy.loc[date]
            S = row['close_spy']
            sigma = row['close_vix']
            q = row['q_yield']
            
            for trade in active_positions:
                # Calcular tiempo hasta vencimiento
                T = trade.time_to_expiry(date_str)
                
                if T > 0:
                    # Obtener tasa risk-free
                    days_remaining = trade.days_to_expiry(date_str)
                    curve_row = {
                        30: row[30],
                        90: row[90],
                        180: row[180],
                        365: row[365]
                    }
                    r = get_risk_free_rate(days_remaining, curve_row)
                    
                    # Calcular griegas del straddle
                    greeks = calculate_straddle_greeks(S, trade.strike, T, r, q, sigma)
                    
                    total_delta += greeks['delta']
                    total_gamma += greeks['gamma']
                    total_vega += greeks['vega']
                    total_theta += greeks['theta']
                    total_rho += greeks['rho']
        
        # Guardar valores del dia
        daily_greeks['date'].append(date)
        daily_greeks['delta'].append(total_delta)
        daily_greeks['gamma'].append(total_gamma)
        daily_greeks['vega'].append(total_vega)
        daily_greeks['theta'].append(total_theta)
        daily_greeks['rho'].append(total_rho)
        daily_greeks['num_positions'].append(len(active_positions))
    
    # Crear DataFrame
    greeks_df = pd.DataFrame(daily_greeks)
    greeks_df.set_index('date', inplace=True)
    
    return greeks_df


# Calcular griegas del portfolio
print("Calculando griegas agregadas del portfolio...")
portfolio_greeks = calculate_portfolio_greeks_timeseries(results, market_data)

print(f"Griegas del portfolio calculadas para {len(portfolio_greeks)} dias de trading")
print(f"\nEstadisticas de las griegas agregadas:")
print(portfolio_greeks.describe())

Calculando griegas agregadas del portfolio...
Griegas del portfolio calculadas para 250 dias de trading

Estadisticas de las griegas agregadas:
            delta       gamma        vega       theta         rho  num_positions
count  250.000000  250.000000  250.000000  250.000000  250.000000     250.000000
mean     0.940195    0.117611    3.312008   -1.984818    0.144803       4.524000
std      1.673488    0.045696    0.804303    0.801228    0.305929       0.683588
min     -3.322722    0.000000    0.000000   -5.917029   -0.714655       1.000000
25%     -0.066261    0.089331    2.874120   -2.288789   -0.012423       4.000000
50%      1.481801    0.118540    3.465912   -1.856957    0.210518       5.000000
75%      2.115888    0.141415    3.843284   -1.517169    0.374239       5.000000
max      3.925663    0.280391    4.625321    0.000000    0.630102       6.000000


## 10. Comparacion de Griegas del Portfolio vs P&L

Analisis de la relacion entre las griegas agregadas del portfolio (suma de todas las posiciones activas) y el P&L de la estrategia.

In [50]:
# 10.1 - Calcular Griegas del Portfolio Agregado

print("=" * 80)
print("APARTADO 10: COMPARACION DE GRIEGAS vs P&L")
print("=" * 80)
print()

def calculate_portfolio_greeks_timeseries(results, market_data):
    """
    Calcula las griegas agregadas del portfolio para cada dia de trading.
    
    Suma las griegas de todas las posiciones activas en cada fecha.
    
    Parameters
    ----------
    results : BacktestResult
        Resultados del backtest
    market_data : pd.DataFrame
        Datos de mercado consolidados
        
    Returns
    -------
    pd.DataFrame
        DataFrame con columnas: delta, gamma, vega, theta, rho, num_positions
    """
    from straddle import calculate_straddle_greeks
    from rates import get_risk_free_rate
    
    # Asegurar que el índice esté en formato datetime
    market_data_copy = market_data.copy()
    if not isinstance(market_data_copy.index, pd.DatetimeIndex):
        market_data_copy.index = pd.to_datetime(market_data_copy.index)
    
    # Inicializar diccionario para guardar griegas diarias
    daily_greeks = {
        'delta': [],
        'gamma': [],
        'vega': [],
        'theta': [],
        'rho': [],
        'num_positions': [],
        'date': []
    }
    
    # Para cada dia de trading
    for date in market_data_copy.index:
        date_str = str(date.date())
        
        # Encontrar todas las posiciones activas en esta fecha
        active_positions = [
            trade for trade in results.trades
            if trade.entry_date <= date_str <= trade.expiry_date
        ]
        
        # Sumar griegas de todas las posiciones activas
        total_delta = 0.0
        total_gamma = 0.0
        total_vega = 0.0
        total_theta = 0.0
        total_rho = 0.0
        
        if active_positions:
            row = market_data_copy.loc[date]
            S = row['close_spy']
            sigma = row['close_vix']
            q = row['q_yield']
            
            for trade in active_positions:
                # Calcular tiempo hasta vencimiento
                T = trade.time_to_expiry(date_str)
                
                if T > 0:
                    # Obtener tasa risk-free
                    days_remaining = trade.days_to_expiry(date_str)
                    curve_row = {
                        30: row[30],
                        90: row[90],
                        180: row[180],
                        365: row[365]
                    }
                    r = get_risk_free_rate(days_remaining, curve_row)
                    
                    # Calcular griegas del straddle
                    greeks = calculate_straddle_greeks(S, trade.strike, T, r, q, sigma)
                    
                    total_delta += greeks['delta']
                    total_gamma += greeks['gamma']
                    total_vega += greeks['vega']
                    total_theta += greeks['theta']
                    total_rho += greeks['rho']
        
        # Guardar valores del dia
        daily_greeks['date'].append(date)
        daily_greeks['delta'].append(total_delta)
        daily_greeks['gamma'].append(total_gamma)
        daily_greeks['vega'].append(total_vega)
        daily_greeks['theta'].append(total_theta)
        daily_greeks['rho'].append(total_rho)
        daily_greeks['num_positions'].append(len(active_positions))
    
    # Crear DataFrame
    greeks_df = pd.DataFrame(daily_greeks)
    greeks_df.set_index('date', inplace=True)
    
    return greeks_df


# Calcular griegas del portfolio
print("Calculando griegas agregadas del portfolio...")
portfolio_greeks = calculate_portfolio_greeks_timeseries(results, market_data)

print(f"Griegas del portfolio calculadas para {len(portfolio_greeks)} dias de trading")
print(f"\nEstadisticas de las griegas agregadas:")
print(portfolio_greeks.describe())

# Calcular cambios diarios para analisis de correlacion
pnl_changes = results.daily_pnl_straddle
greeks_changes = portfolio_greeks[['delta', 'gamma', 'vega', 'theta', 'rho']].diff()

# Combinar en un solo DataFrame para analisis
combined_df = pd.DataFrame({
    'pnl_change': pnl_changes,
    'delta_change': greeks_changes['delta'],
    'gamma_change': greeks_changes['gamma'],
    'vega_change': greeks_changes['vega'],
    'theta_change': greeks_changes['theta'],
    'rho_change': greeks_changes['rho']
}).dropna()

print(f"\nCambios diarios calculados: {len(combined_df)} observaciones")

APARTADO 10: COMPARACION DE GRIEGAS vs P&L

Calculando griegas agregadas del portfolio...
Griegas del portfolio calculadas para 250 dias de trading

Estadisticas de las griegas agregadas:
            delta       gamma        vega       theta         rho  num_positions
count  250.000000  250.000000  250.000000  250.000000  250.000000     250.000000
mean     0.940195    0.117611    3.312008   -1.984818    0.144803       4.524000
std      1.673488    0.045696    0.804303    0.801228    0.305929       0.683588
min     -3.322722    0.000000    0.000000   -5.917029   -0.714655       1.000000
25%     -0.066261    0.089331    2.874120   -2.288789   -0.012423       4.000000
50%      1.481801    0.118540    3.465912   -1.856957    0.210518       5.000000
75%      2.115888    0.141415    3.843284   -1.517169    0.374239       5.000000
max      3.925663    0.280391    4.625321    0.000000    0.630102       6.000000

Cambios diarios calculados: 0 observaciones


In [48]:
# 10.2 - Visualizaciones: Griegas vs P&L (NO HEDGED)

print("\n" + "=" * 80)
print("10.2 VISUALIZACIONES: GRIEGAS vs P&L - ESTRATEGIA NO HEDGED")
print("=" * 80)
print()

# Crear visualizaciones comparativas en un solo dashboard
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Delta vs P&L Acumulado',
        'Gamma vs P&L Acumulado',
        'Vega vs P&L Acumulado',
        'Theta vs P&L Acumulado',
        'Correlacion: Cambios en Griegas vs Cambios en P&L',
        'Distribucion de P&L por Rango de Delta'
    ),
    specs=[
        [{"secondary_y": True}, {"secondary_y": True}],
        [{"secondary_y": True}, {"secondary_y": True}],
        [{"type": "bar"}, {"type": "box"}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.12,
    row_heights=[0.3, 0.3, 0.4]
)

# Colores
color_pnl = '#2E86AB'
color_delta = '#6A4C93'
color_gamma = '#A23B72'
color_vega = '#F18F01'
color_theta = '#C73E1D'

# (1,1) Delta vs P&L
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=results.cumulative_pnl_straddle.values,
        name='P&L',
        line=dict(color=color_pnl, width=2),
        hovertemplate='<b>P&L:</b> $%{y:.2f}<extra></extra>'
    ),
    row=1, col=1, secondary_y=False
)
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=portfolio_greeks['delta'],
        name='Delta',
        line=dict(color=color_delta, width=2, dash='dot'),
        hovertemplate='<b>Delta:</b> %{y:.2f}<extra></extra>'
    ),
    row=1, col=1, secondary_y=True
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3, row=1, col=1, secondary_y=False)

# (1,2) Gamma vs P&L
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=results.cumulative_pnl_straddle.values,
        name='P&L',
        line=dict(color=color_pnl, width=2),
        showlegend=False,
        hovertemplate='<b>P&L:</b> $%{y:.2f}<extra></extra>'
    ),
    row=1, col=2, secondary_y=False
)
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=portfolio_greeks['gamma'],
        name='Gamma',
        line=dict(color=color_gamma, width=2, dash='dot'),
        hovertemplate='<b>Gamma:</b> %{y:.2f}<extra></extra>'
    ),
    row=1, col=2, secondary_y=True
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3, row=1, col=2, secondary_y=False)

# (2,1) Vega vs P&L
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=results.cumulative_pnl_straddle.values,
        name='P&L',
        line=dict(color=color_pnl, width=2),
        showlegend=False,
        hovertemplate='<b>P&L:</b> $%{y:.2f}<extra></extra>'
    ),
    row=2, col=1, secondary_y=False
)
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=portfolio_greeks['vega'],
        name='Vega',
        line=dict(color=color_vega, width=2, dash='dot'),
        hovertemplate='<b>Vega:</b> %{y:.2f}<extra></extra>'
    ),
    row=2, col=1, secondary_y=True
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3, row=2, col=1, secondary_y=False)

# (2,2) Theta vs P&L
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=results.cumulative_pnl_straddle.values,
        name='P&L',
        line=dict(color=color_pnl, width=2),
        showlegend=False,
        hovertemplate='<b>P&L:</b> $%{y:.2f}<extra></extra>'
    ),
    row=2, col=2, secondary_y=False
)
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=portfolio_greeks['theta'],
        name='Theta',
        line=dict(color=color_theta, width=2, dash='dot'),
        hovertemplate='<b>Theta:</b> %{y:.2f}<extra></extra>'
    ),
    row=2, col=2, secondary_y=True
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3, row=2, col=2, secondary_y=False)

# (3,1) Correlaciones
correlations = {
    'Delta': combined_df['pnl_change'].corr(combined_df['delta_change']),
    'Gamma': combined_df['pnl_change'].corr(combined_df['gamma_change']),
    'Vega': combined_df['pnl_change'].corr(combined_df['vega_change']),
    'Theta': combined_df['pnl_change'].corr(combined_df['theta_change']),
    'Rho': combined_df['pnl_change'].corr(combined_df['rho_change'])
}

colors_corr = [color_delta, color_gamma, color_vega, color_theta, '#555555']
fig.add_trace(
    go.Bar(
        x=list(correlations.keys()),
        y=list(correlations.values()),
        name='Correlacion',
        marker_color=colors_corr,
        text=[f'{v:+.3f}' for v in correlations.values()],
        textposition='outside',
        hovertemplate='<b>%{x}:</b> %{y:.4f}<extra></extra>'
    ),
    row=3, col=1
)
fig.add_hline(y=0, line_dash="solid", line_color="black", line_width=1, row=3, col=1)

# (3,2) Box plot de P&L por rango de Delta
# Categorizar por Delta
delta_categories = pd.cut(
    portfolio_greeks['delta'], 
    bins=[-np.inf, -10, -5, 0, 5, 10, np.inf],
    labels=['<-10', '-10 a -5', '-5 a 0', '0 a 5', '5 a 10', '>10']
)

# Obtener P&L diario para cada categoria
pnl_by_delta = {}
for cat in delta_categories.unique():
    if pd.notna(cat):
        mask = delta_categories == cat
        pnl_by_delta[cat] = results.daily_pnl_straddle[mask].values

for cat, pnl_vals in pnl_by_delta.items():
    fig.add_trace(
        go.Box(
            y=pnl_vals,
            name=str(cat),
            boxmean='sd',
            hovertemplate='<b>Delta Range:</b> %{x}<br><b>P&L:</b> $%{y:.2f}<extra></extra>'
        ),
        row=3, col=2
    )

# Actualizar ejes
fig.update_yaxes(title_text="P&L ($)", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Delta", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="P&L ($)", row=1, col=2, secondary_y=False)
fig.update_yaxes(title_text="Gamma", row=1, col=2, secondary_y=True)
fig.update_yaxes(title_text="P&L ($)", row=2, col=1, secondary_y=False)
fig.update_yaxes(title_text="Vega", row=2, col=1, secondary_y=True)
fig.update_yaxes(title_text="P&L ($)", row=2, col=2, secondary_y=False)
fig.update_yaxes(title_text="Theta", row=2, col=2, secondary_y=True)
fig.update_yaxes(title_text="Correlacion", row=3, col=1)
fig.update_yaxes(title_text="P&L Diario ($)", row=3, col=2)
fig.update_xaxes(title_text="Griega", row=3, col=1)
fig.update_xaxes(title_text="Rango de Delta", row=3, col=2)

fig.update_layout(
    height=1200,
    showlegend=True,
    template='plotly_white',
    title_text='Dashboard Comparativo: Griegas vs P&L - Estrategia NO HEDGED',
    title_x=0.5,
    title_font_size=16,
    hovermode='x unified',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.05,
        xanchor="center",
        x=0.5
    )
)

fig.show()

# Resumen estadistico
print("\n" + "=" * 80)
print("RESUMEN ESTADISTICO: GRIEGAS vs P&L (NO HEDGED)")
print("=" * 80)
print(f"\n1. CORRELACIONES (Cambios Diarios)")
print("-" * 80)
for greek_name, corr_val in correlations.items():
    significance = "ALTA" if abs(corr_val) > 0.5 else "MEDIA" if abs(corr_val) > 0.3 else "BAJA"
    print(f"   {greek_name:8s}: {corr_val:+.4f}  [{significance}]")

print(f"\n2. METRICAS DEL PORTFOLIO")
print("-" * 80)
print(f"   Delta promedio:          {portfolio_greeks['delta'].mean():+8.2f}")
print(f"   Delta std:               {portfolio_greeks['delta'].std():8.2f}")
print(f"   Gamma promedio:          {portfolio_greeks['gamma'].mean():8.2f}")
print(f"   Vega promedio:           {portfolio_greeks['vega'].mean():8.2f}")
print(f"   Theta promedio:          {portfolio_greeks['theta'].mean():+8.2f}")
print(f"   Posiciones activas (avg):{portfolio_greeks['num_positions'].mean():8.1f}")

print(f"\n3. RESULTADOS P&L")
print("-" * 80)
print(f"   P&L Total:               ${results.summary['total_pnl']:+.2f}")
print(f"   P&L por Trade:           ${results.summary['avg_pnl_per_trade']:+.2f}")
print(f"   Win Rate:                {results.summary['win_rate']*100:.1f}%")
print(f"   Max Drawdown:            ${results.summary['max_drawdown']:+.2f}")
print(f"   Profit Factor:           {results.summary['profit_factor']:.2f}")
print("=" * 80)


10.2 VISUALIZACIONES: GRIEGAS vs P&L - ESTRATEGIA NO HEDGED




RESUMEN ESTADISTICO: GRIEGAS vs P&L (NO HEDGED)

1. CORRELACIONES (Cambios Diarios)
--------------------------------------------------------------------------------
   Delta   : +nan  [BAJA]
   Gamma   : +nan  [BAJA]
   Vega    : +nan  [BAJA]
   Theta   : +nan  [BAJA]
   Rho     : +nan  [BAJA]

2. METRICAS DEL PORTFOLIO
--------------------------------------------------------------------------------
   Delta promedio:             +0.94
   Delta std:                   1.67
   Gamma promedio:              0.12
   Vega promedio:               3.31
   Theta promedio:             -1.98
   Posiciones activas (avg):     4.5

3. RESULTADOS P&L
--------------------------------------------------------------------------------
   P&L Total:               $-2461.69
   P&L por Trade:           $-4.96
   Win Rate:                34.0%
   Max Drawdown:            $-2468.01
   Profit Factor:           0.38


In [49]:
# 10.2 Simulacion de Estrategia HEDGED y Comparacion

print("\n" + "=" * 80)
print("10.2 ESTRATEGIA DELTA HEDGED - Simulacion")
print("=" * 80)
print()

# Simular delta hedging: comprar/vender SPY para neutralizar delta
# Costo de hedge = Delta * S * comision_estimada

def simulate_delta_hedging(portfolio_greeks, market_data, commission_rate=0.001):
    """
    Simula el costo de delta hedging comprando/vendiendo SPY.
    
    Parameters
    ----------
    portfolio_greeks : pd.DataFrame
        Griegas del portfolio agregado
    market_data : pd.DataFrame
        Datos de mercado con precios de SPY
    commission_rate : float
        Tasa de comision para ajustes (default 0.1%)
        
    Returns
    -------
    pd.DataFrame
        DataFrame con: delta_position (shares de SPY), hedge_cost, cumulative_hedge_cost
    """
    hedge_data = []
    cumulative_cost = 0.0
    previous_delta_position = 0.0
    
    # Asegurar indices alineados
    market_data_copy = market_data.copy()
    if not isinstance(market_data_copy.index, pd.DatetimeIndex):
        market_data_copy.index = pd.to_datetime(market_data_copy.index)
    
    for date in portfolio_greeks.index:
        delta = portfolio_greeks.loc[date, 'delta']
        spot = market_data_copy.loc[date, 'close_spy']
        
        # Posicion necesaria en SPY para neutralizar delta (vender delta shares)
        delta_position = -delta  # Negativo porque vendemos para neutralizar
        
        # Cambio en la posicion (rebalanceo)
        position_change = abs(delta_position - previous_delta_position)
        
        # Costo del rebalanceo (comision sobre el notional)
        daily_cost = position_change * spot * commission_rate
        cumulative_cost += daily_cost
        
        hedge_data.append({
            'date': date,
            'delta': delta,
            'delta_position': delta_position,
            'spot': spot,
            'position_change': position_change,
            'hedge_cost': daily_cost,
            'cumulative_hedge_cost': cumulative_cost
        })
        
        previous_delta_position = delta_position
    
    return pd.DataFrame(hedge_data).set_index('date')


# Calcular costos de hedging
print("Calculando costos de delta hedging...")
hedge_df = simulate_delta_hedging(portfolio_greeks, market_data, commission_rate=0.0005)

# P&L de la estrategia hedged = P&L straddle - costos de hedging
pnl_hedged = results.cumulative_pnl_straddle - hedge_df['cumulative_hedge_cost']

print(f"Costos totales de hedging: ${hedge_df['cumulative_hedge_cost'].iloc[-1]:.2f}")
print(f"P&L Final NO HEDGED: ${results.cumulative_pnl_straddle.iloc[-1]:.2f}")
print(f"P&L Final HEDGED: ${pnl_hedged.iloc[-1]:.2f}")
print()

# Visualizacion comparativa HEDGED vs NO HEDGED
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'P&L Acumulado: Hedged vs No Hedged',
        'Delta del Portfolio: Hedged vs No Hedged',
        'Costos Acumulados de Delta Hedging',
        'Posicion en SPY para Delta Hedging',
        'Comparacion de Volatilidad de P&L',
        'Metricas Comparativas'
    ),
    specs=[
        [{"secondary_y": False}, {"secondary_y": False}],
        [{"secondary_y": False}, {"secondary_y": False}],
        [{"type": "bar"}, {"type": "table"}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.12,
    row_heights=[0.3, 0.3, 0.4]
)

# (1,1) P&L Comparativo
fig.add_trace(
    go.Scatter(
        x=results.cumulative_pnl_straddle.index,
        y=results.cumulative_pnl_straddle.values,
        name='No Hedged',
        line=dict(color='#2E86AB', width=2.5),
        hovertemplate='<b>No Hedged P&L:</b> $%{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=pnl_hedged.index,
        y=pnl_hedged.values,
        name='Delta Hedged',
        line=dict(color='#06A77D', width=2.5, dash='dash'),
        hovertemplate='<b>Hedged P&L:</b> $%{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)
fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.5, row=1, col=1)

# (1,2) Delta Comparativo
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=portfolio_greeks['delta'],
        name='Delta No Hedged',
        line=dict(color='#6A4C93', width=2),
        hovertemplate='<b>Delta:</b> %{y:.2f}<extra></extra>'
    ),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(
        x=portfolio_greeks.index,
        y=[0] * len(portfolio_greeks),
        name='Delta Hedged (target)',
        line=dict(color='#228B22', width=2, dash='dash'),
        hovertemplate='<b>Target Delta:</b> 0<extra></extra>'
    ),
    row=1, col=2
)
fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.5, row=1, col=2)

# (2,1) Costos de Hedging
fig.add_trace(
    go.Scatter(
        x=hedge_df.index,
        y=hedge_df['cumulative_hedge_cost'],
        name='Costos Acumulados',
        line=dict(color='#C73E1D', width=2),
        fill='tozeroy',
        fillcolor='rgba(199, 62, 29, 0.2)',
        hovertemplate='<b>Costo Acumulado:</b> $%{y:.2f}<extra></extra>'
    ),
    row=2, col=1
)

# (2,2) Posicion en SPY
fig.add_trace(
    go.Scatter(
        x=hedge_df.index,
        y=hedge_df['delta_position'],
        name='Posicion SPY',
        line=dict(color='#F18F01', width=2),
        fill='tozeroy',
        fillcolor='rgba(241, 143, 1, 0.2)',
        hovertemplate='<b>Shares SPY:</b> %{y:.2f}<extra></extra>'
    ),
    row=2, col=2
)
fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.5, row=2, col=2)

# (3,1) Comparacion de volatilidad
daily_pnl_no_hedged = results.daily_pnl_straddle
daily_pnl_hedged = pnl_hedged.diff().fillna(0)

vol_comparison = {
    'No Hedged': [
        daily_pnl_no_hedged.std(),
        daily_pnl_no_hedged.mean(),
        daily_pnl_no_hedged.min(),
        daily_pnl_no_hedged.max()
    ],
    'Hedged': [
        daily_pnl_hedged.std(),
        daily_pnl_hedged.mean(),
        daily_pnl_hedged.min(),
        daily_pnl_hedged.max()
    ]
}

x_metrics = ['Volatilidad', 'Media', 'Min', 'Max']
fig.add_trace(
    go.Bar(
        x=x_metrics,
        y=vol_comparison['No Hedged'],
        name='No Hedged',
        marker_color='#2E86AB',
        text=[f'${v:.2f}' for v in vol_comparison['No Hedged']],
        textposition='outside'
    ),
    row=3, col=1
)
fig.add_trace(
    go.Bar(
        x=x_metrics,
        y=vol_comparison['Hedged'],
        name='Hedged',
        marker_color='#06A77D',
        text=[f'${v:.2f}' for v in vol_comparison['Hedged']],
        textposition='outside'
    ),
    row=3, col=1
)

# (3,2) Tabla comparativa
metrics_comparison = [
    ['Metrica', 'No Hedged', 'Hedged', 'Diferencia'],
    ['P&L Final', 
     f"${results.cumulative_pnl_straddle.iloc[-1]:.2f}",
     f"${pnl_hedged.iloc[-1]:.2f}",
     f"${pnl_hedged.iloc[-1] - results.cumulative_pnl_straddle.iloc[-1]:.2f}"],
    ['Volatilidad Diaria',
     f"${daily_pnl_no_hedged.std():.2f}",
     f"${daily_pnl_hedged.std():.2f}",
     f"{((daily_pnl_hedged.std() / daily_pnl_no_hedged.std() - 1) * 100):.1f}%"],
    ['Max Drawdown',
     f"${results.cumulative_pnl_straddle.min():.2f}",
     f"${pnl_hedged.min():.2f}",
     f"${pnl_hedged.min() - results.cumulative_pnl_straddle.min():.2f}"],
    ['Sharpe Ratio (est)',
     f"{(daily_pnl_no_hedged.mean() / daily_pnl_no_hedged.std() * np.sqrt(252)):.2f}",
     f"{(daily_pnl_hedged.mean() / daily_pnl_hedged.std() * np.sqrt(252)):.2f}",
     f"{((daily_pnl_hedged.mean() / daily_pnl_hedged.std()) - (daily_pnl_no_hedged.mean() / daily_pnl_no_hedged.std())):.2f}"],
    ['Costo de Hedging',
     '$0.00',
     f"${hedge_df['cumulative_hedge_cost'].iloc[-1]:.2f}",
     f"-${hedge_df['cumulative_hedge_cost'].iloc[-1]:.2f}"]
]

fig.add_trace(
    go.Table(
        header=dict(
            values=metrics_comparison[0],
            fill_color='#333333',
            font=dict(color='white', size=12),
            align='left'
        ),
        cells=dict(
            values=list(zip(*metrics_comparison[1:])),
            fill_color=[['#f0f0f0', 'white'] * 3],
            align='left',
            font=dict(size=11)
        )
    ),
    row=3, col=2
)

# Actualizar ejes
fig.update_xaxes(title_text="Fecha", row=1, col=1)
fig.update_xaxes(title_text="Fecha", row=1, col=2)
fig.update_xaxes(title_text="Fecha", row=2, col=1)
fig.update_xaxes(title_text="Fecha", row=2, col=2)
fig.update_xaxes(title_text="Metrica", row=3, col=1)

fig.update_yaxes(title_text="P&L ($)", row=1, col=1)
fig.update_yaxes(title_text="Delta", row=1, col=2)
fig.update_yaxes(title_text="Costo ($)", row=2, col=1)
fig.update_yaxes(title_text="Shares", row=2, col=2)
fig.update_yaxes(title_text="Valor ($)", row=3, col=1)

fig.update_layout(
    height=1400,
    showlegend=True,
    template='plotly_white',
    title_text='Comparacion Completa: Estrategia HEDGED vs NO HEDGED',
    title_x=0.5,
    title_font_size=16,
    hovermode='x unified',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.05,
        xanchor="center",
        x=0.5
    )
)

fig.show()

# Resumen final
print("\n" + "=" * 80)
print("RESUMEN COMPARATIVO FINAL")
print("=" * 80)
print(f"\nESTRATEGIA NO HEDGED:")
print(f"  - P&L Final:           ${results.cumulative_pnl_straddle.iloc[-1]:+.2f}")
print(f"  - Volatilidad Diaria:  ${daily_pnl_no_hedged.std():.2f}")
print(f"  - Delta Promedio:      {portfolio_greeks['delta'].mean():+.2f}")
print(f"  - Max Drawdown:        ${results.cumulative_pnl_straddle.min():+.2f}")

print(f"\nESTRATEGIA DELTA HEDGED:")
print(f"  - P&L Final:           ${pnl_hedged.iloc[-1]:+.2f}")
print(f"  - Volatilidad Diaria:  ${daily_pnl_hedged.std():.2f}")
print(f"  - Costo de Hedging:    ${hedge_df['cumulative_hedge_cost'].iloc[-1]:.2f}")
print(f"  - Max Drawdown:        ${pnl_hedged.min():+.2f}")

print(f"\nIMPACTO DEL HEDGING:")
print(f"  - Reduccion de P&L:    ${pnl_hedged.iloc[-1] - results.cumulative_pnl_straddle.iloc[-1]:+.2f}")
print(f"  - Reduccion de Vol:    {((daily_pnl_hedged.std() / daily_pnl_no_hedged.std() - 1) * 100):+.1f}%")
print(f"  - Mejora en Sharpe:    {((daily_pnl_hedged.mean() / daily_pnl_hedged.std()) - (daily_pnl_no_hedged.mean() / daily_pnl_no_hedged.std())):+.2f}")

print("\n" + "=" * 80)
print("NOTA: Esta es una simulacion simplificada de delta hedging.")
print("Asume rebalanceo diario con comisiones del 0.05% sobre el notional.")
print("En una implementacion real, habria costos de slippage, bid-ask spreads,")
print("y posiblemente diferentes frecuencias de rebalanceo.")
print("=" * 80)


10.2 ESTRATEGIA DELTA HEDGED - Simulacion

Calculando costos de delta hedging...
Costos totales de hedging: $43.43
P&L Final NO HEDGED: $-2461.69
P&L Final HEDGED: $nan



C:\Users\diego\AppData\Local\Temp\ipykernel_55156\1716743410.py:235: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\diego\AppData\Local\Temp\ipykernel_55156\1716743410.py:236: RuntimeWarning:

invalid value encountered in scalar divide




RESUMEN COMPARATIVO FINAL

ESTRATEGIA NO HEDGED:
  - P&L Final:           $-2461.69
  - Volatilidad Diaria:  $29.81
  - Delta Promedio:      +0.94
  - Max Drawdown:        $-2461.69

ESTRATEGIA DELTA HEDGED:
  - P&L Final:           $+nan
  - Volatilidad Diaria:  $0.00
  - Costo de Hedging:    $43.43
  - Max Drawdown:        $+nan

IMPACTO DEL HEDGING:
  - Reduccion de P&L:    $+nan
  - Reduccion de Vol:    -100.0%
  - Mejora en Sharpe:    +nan

NOTA: Esta es una simulacion simplificada de delta hedging.
Asume rebalanceo diario con comisiones del 0.05% sobre el notional.
En una implementacion real, habria costos de slippage, bid-ask spreads,
y posiblemente diferentes frecuencias de rebalanceo.


C:\Users\diego\AppData\Local\Temp\ipykernel_55156\1716743410.py:312: RuntimeWarning:

invalid value encountered in scalar divide

